In [124]:
#%pip install ucimlrepo
#%pip install scikit-learn
from ucimlrepo import fetch_ucirepo 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error 

online_retail = fetch_ucirepo(id=352) 

df = pd.DataFrame(online_retail.data.features)

print("Rows:",df.shape[0]) 
print("Total Features:", df.shape[1])
print("Columns:")
df.head()

Rows: 541909
Total Features: 6
Columns:


,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [125]:
print("Unique Countries:")
print(df['Country'].unique())

Unique Countries:
['United Kingdom' 'France' 'Australia' 'Netherlands' 'Germany' 'Norway'
 'EIRE' 'Switzerland' 'Spain' 'Poland' 'Portugal' 'Italy' 'Belgium'
 'Lithuania' 'Japan' 'Iceland' 'Channel Islands' 'Denmark' 'Cyprus'
 'Sweden' 'Austria' 'Israel' 'Finland' 'Bahrain' 'Greece' 'Hong Kong'
 'Singapore' 'Lebanon' 'United Arab Emirates' 'Saudi Arabia'
 'Czech Republic' 'Canada' 'Unspecified' 'Brazil' 'USA'
 'European Community' 'Malta' 'RSA']


In [126]:
df[df['Quantity'] < 0]['Quantity'].count()

np.int64(10624)

In [127]:
df[df['UnitPrice'] < 0]['UnitPrice'].count()

np.int64(2)

In [128]:
df_clean = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df_clean[df_clean['Quantity'] < 0]['Quantity'].count()

np.int64(0)

In [129]:
df_clean[df_clean['UnitPrice'] < 0]['UnitPrice'].count()

np.int64(0)

In [130]:
data_clean = df_clean[['Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'Country']]

In [131]:
data_clean.head()

,Description,Quantity,InvoiceDate,UnitPrice,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,United Kingdom
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,United Kingdom
2,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,United Kingdom
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,United Kingdom
4,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,United Kingdom


In [132]:
data_clean.isnull().sum()

Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
Country        0
dtype: int64

In [133]:
print("🔍 AUDITING FOR HIDDEN PLACEHOLDERS...\n")

# 1. Search for hidden text placeholders in ALL columns at once
text_placeholders = ['unspecified', 'unknown', 'none', 'null', 'missing']

for placeholder in text_placeholders:
    # Check if the placeholder text exists anywhere in the dataframe (case-insensitive)
    matches = data_clean.apply(lambda col: col.astype(str).str.lower().str.contains(placeholder)).sum().sum()
    if matches > 0:
        print(f"⚠️ Found {matches} instances of the text '{placeholder}' hidden inside your columns!")
    else:
        print(f"✅ Clean: No instances of '{placeholder}' found.")

print("\n🔢 AUDITING FOR FAKE NUMERICAL ID NUMBERS...")

# 2. Check if there are any suspicious fake numeric IDs or entries
# Since Quantity and UnitPrice must be > 0, let's verify no extreme placeholder outliers exist
zero_count = (data_clean[['Quantity', 'UnitPrice']] == 0).sum().sum()
print(f"✅ Clean: Found {zero_count} exact zeros in Quantity or UnitPrice.")

max_qty = data_clean['Quantity'].max()
max_price = data_clean['UnitPrice'].max()
print(f"📊 Maximum structural values found: Max Quantity = {max_qty:,} | Max Unit Price = ${max_price:,.2f}")

🔍 AUDITING FOR HIDDEN PLACEHOLDERS...

⚠️ Found 446 instances of the text 'unspecified' hidden inside your columns!
✅ Clean: No instances of 'unknown' found.
✅ Clean: No instances of 'none' found.
✅ Clean: No instances of 'null' found.
✅ Clean: No instances of 'missing' found.

🔢 AUDITING FOR FAKE NUMERICAL ID NUMBERS...
✅ Clean: Found 0 exact zeros in Quantity or UnitPrice.
📊 Maximum structural values found: Max Quantity = 80,995 | Max Unit Price = $13,541.33


In [134]:
data_clean[data_clean['Description'].astype(str).str.lower() == 'unspecified']['Description'].count()

np.int64(0)

In [135]:
data_clean[data_clean['Quantity'].astype(str).str.lower()  == 'unspecified']['Quantity'].count()

np.int64(0)

In [136]:
data_clean[data_clean['InvoiceDate'].astype(str).str.lower()  == 'unspecified']['InvoiceDate'].count()

np.int64(0)

In [137]:
data_clean[data_clean['UnitPrice'].astype(str).str.lower()  == 'unspecified']['UnitPrice'].count()

np.int64(0)

In [138]:
data_clean[data_clean['Country'].astype(str).str.lower()  == 'unspecified']['Country'].count()

np.int64(446)

In [139]:
data_final = data_clean[data_clean['Country'].astype(str).str.lower()  != 'unspecified']
data_final[data_final['Country'].astype(str).str.lower()  == 'unspecified']['Country'].count()

np.int64(0)

In [140]:
#data_final.groupby('Country')['Quantity'*'UnitPrice'].sum().sort_values(ascending=False).head(10)

In [141]:
data_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 529658 entries, 0 to 541908
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Description  529658 non-null  object 
 1   Quantity     529658 non-null  int64  
 2   InvoiceDate  529658 non-null  object 
 3   UnitPrice    529658 non-null  float64
 4   Country      529658 non-null  object 
dtypes: float64(1), int64(1), object(3)
memory usage: 24.2+ MB


In [142]:
data_final.assign(Revenue = data_final['Quantity'] * data_final['UnitPrice']).groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)

Country
United Kingdom    9025222.084
Netherlands        285446.340
EIRE               283453.960
Germany            228867.140
France             209715.110
Australia          138521.310
Spain               61577.110
Switzerland         57089.900
Belgium             41196.340
Sweden              38378.330
Name: Revenue, dtype: float64

In [143]:
data_final['Target_Revenue'] = data_final['Quantity'] * data_final['UnitPrice']

/var/folders/74/87zgzy_911q164s59j3jb5nr0000gn/T/ipykernel_24177/1521744966.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_final['Target_Revenue'] = data_final['Quantity'] * data_final['UnitPrice']


In [144]:
country_0 = data_final[data_final['Country'].astype(str).str.lower()  == 'united kingdom'].copy()
country_1 = data_final[data_final['Country'].astype(str).str.lower()  == 'netherlands'].copy()
country_2 = data_final[data_final['Country'].astype(str).str.lower()  == 'eire'].copy()

In [145]:
def train_and_evaluate_region(region_df, region_name):
    # Isolate your features (X) and target (y)
    X = region_df[['Quantity', 'UnitPrice']]
    y = region_df['Target_Revenue']
    
    # 2.1 Split the data into a training set and validation set at a ratio of 75:25
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
    
    # 2.2 Train the model and make predictions for the validation set
    model = LinearRegression()
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    
    # 2.3 Save the predictions and correct answers for the validation set
    results = pd.DataFrame({
        'true_values': y_val,
        'predictions': predictions
    })
    
    # 2.4 Calculate the average volume of predicted reserves (revenue) and model RMSE
    avg_predicted_revenue = results['predictions'].mean()
    rmse = np.sqrt(mean_squared_error(results['true_values'], results['predictions']))
    
    # Print metrics exactly as requested in your instructions
    print(f"📊 {region_name} Model Evaluation:")
    print(f"    Average Predicted Transaction Value: ${avg_predicted_revenue:,.2f}")
    print(f"    Model RMSE: ${rmse:,.2f}\n")
    
    return results

# Run the machine learning model loops!
results_r0 = train_and_evaluate_region(country_0, "Region 0 (United Kingdom)")
results_r1 = train_and_evaluate_region(country_1, "Region 1 (Netherlands)")
results_r2 = train_and_evaluate_region(country_2, "Region 2 (EIRE)")

📊 Region 0 (United Kingdom) Model Evaluation:
    Average Predicted Transaction Value: $18.54
    Model RMSE: $57.36

📊 Region 1 (Netherlands) Model Evaluation:
    Average Predicted Transaction Value: $119.87
    Model RMSE: $99.60

📊 Region 2 (EIRE) Model Evaluation:
    Average Predicted Transaction Value: $35.72
    Model RMSE: $45.30



Region 2 has the lowest RMSE with quite a high Average Predicted Trasaction Value. This is our best choice of the three options. Though Region 1 has a high Average Predicted Transaction Value, it has a very high Model RMSE, which is not ideal. EIRE in this case it our top selection for our high-stakes market, and we can plan a physical, high-capital infrastructure expansion with this location.